# 1. Section: Run the Simulation Script
We will execute the `main.py` script directly from the notebook. 
*(Note: To save time during testing, we are running a smaller set of capacities/days via the imported function rather than the full 150k days shell command).*

In [1]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import autograd.numpy as np

# Import the simulation runner
from main import run_simulation

# Run a quick demonstration version of the simulation
capacities = [4000, 8000, 32000]
number_days = [100, 100, 100] # Reduced for quick notebook execution

for i, cap in enumerate(capacities):
    print(f"\nRunning simulation for TNC capacity = {cap}")
    run_simulation(
        tnc_capacity=cap,
        output_dir=f"./2-Results/tnc_capacity_{cap}",
        number_days=number_days[i]
    )

NameError: name 'Travelers' is not defined

# 2. Section: Load Simulation Results
Use the `json` and `os` modules to read `final_results.json` from the output directories. Parse the final allocations, hyperparameters, and profit metrics into a pandas DataFrame for structured analysis.

In [ ]:
results_data = []
base_dir = "./2-Results"

if os.path.exists(base_dir):
    for folder in os.listdir(base_dir):
        if folder.startswith("tnc_capacity_"):
            json_path = os.path.join(base_dir, folder, "final_results.json")
            if os.path.exists(json_path):
                with open(json_path, "r") as f:
                    data = json.load(f)
                    
                    # Flatten the data for the DataFrame
                    flat_data = {
                        "Capacity": data["tnc_capacity"],
                        "TNC_Fare": data["tnc_params"]["fare"],
                        "TNC_CapRatio": data["tnc_params"]["capacity_ratio_to_MaaS"],
                        "MaaS_Fare": data["maas_params"]["fare"],
                        "MaaS_ShareTNC": data["maas_params"]["share_TNC"],
                        "TNC_Profit": data["profits"]["tnc_profit"],
                        "MaaS_Profit": data["profits"]["maas_profit"]
                    }
                    results_data.append(flat_data)

df_results = pd.DataFrame(results_data).sort_values(by="Capacity").reset_index(drop=True)
display(df_results)

# 3. Section: Visualize Allocation Evolution
Load the allocation arrays by traveler type and plot the modal split over the simulation days using matplotlib to evaluate the system's equilibrium state. 

*(Since `allocation_history` and `allocation_by_type` are currently only plotted and not exported to JSON by `main.py`, we will display the previously saved PNG charts.)*

In [ ]:
from IPython.display import Image, display

# Display the allocation evolution plots generated by the simulation
for cap in capacities:
    img_path = f"./2-Results/tnc_capacity_{cap}/total_allocation.png"
    if os.path.exists(img_path):
        print(f"Total Allocation Evolution for Capacity: {cap}")
        display(Image(filename=img_path))
    
    img_path_type = f"./2-Results/tnc_capacity_{cap}/allocation_by_type.png"
    if os.path.exists(img_path_type):
        print(f"Allocation By Type for Capacity: {cap}")
        display(Image(filename=img_path_type))

# 4. Section: Analyze Profitability
Extract the profits dictionary from the parsed JSON data. Plot a bar chart comparing TNC and MaaS net incomes against varying total TNC capacities.

In [ ]:
if not df_results.empty:
    fig, ax = plt.subplots(figsize=(8, 5))
    
    x = np.arange(len(df_results["Capacity"]))
    width = 0.35

    ax.bar(x - width/2, df_results["TNC_Profit"], width, label='TNC Profit', color='skyblue')
    ax.bar(x + width/2, df_results["MaaS_Profit"], width, label='MaaS Profit', color='salmon')

    ax.set_ylabel('Net Income ($)')
    ax.set_xlabel('TNC Capacity (veh*km)')
    ax.set_title('Profitability Comparison: TNC vs MaaS')
    ax.set_xticks(x)
    ax.set_xticklabels(df_results["Capacity"])
    ax.legend()
    ax.grid(axis='y', linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.show()
else:
    print("No result data available to plot.")

# 5. Section: Verify Gradient Computations
Import the TNC, MaaS, and Travelers classes. Reconstruct the utility equations and evaluate the output of `check_gradients()` to ensure the analytical derivatives strictly match the autograd calculations.

In [ ]:
from entities import TNC, MT, MaaS, Travelers
from main import compute_utilities, check_gradients

# Re-instantiate test objects for gradient verification
travelers = [
    Travelers(number_traveler=200, trip_length=30, value_time=25, value_wait=50),
    Travelers(number_traveler=150, trip_length=10, value_time=30, value_wait=60)
]

tnc = TNC(ASC=10.0, fare=3, detour_ratio=1.3, average_speed=40, average_veh_travel_dist_per_day=320,
          capacity_ratio_to_MaaS=0.4, total_service_capacity=4000, 
          trip_length_per_traveler_type=[t.trip_length for t in travelers],
          value_waiting_time_per_traveler_type=[t.value_wait for t in travelers],
          cost_purchasing_capacity_TNC=10, operating_cost=10, lambda_T=1.0)

mt = MT(ASC=0.0, fare=9, n_transfer_per_length=0.3, detour_ratio=1.8, 
        average_speed=15, access_time=1/6, transit_time=1/12)

maas = MaaS(ASC=5.0, fare=2, share_TNC=0.2, detour_ratio_TNC=1.3, average_speed_TNC=40,
            capacity_ratio_from_TNC=0.4, total_service_capacity_TNC=4000,
            average_veh_travel_dist_per_day_TNC=320, cost_purchasing_capacity_TNC=10,
            trip_length_per_traveler_type=[t.trip_length for t in travelers],
            value_travel_time_per_traveler_type=[t.value_time for t in travelers],
            value_waiting_time_per_traveler_type=[t.value_wait for t in travelers],
            detour_ratio_MT=1.8, average_speed_MT=15, transit_time_MT=1/12,
            n_transfer_per_length_MT=0.3, cost_purchasing_capacity_MT=5, lambda_M=1.0)

services = [tnc, mt, maas]

# Fake an allocation layout
allocation = {"TNC": [100, 75], "MT": [50, 30], "MaaS": [50, 45]}
tnc.get_allocation(allocation)
maas.get_allocation(allocation)

utilities = compute_utilities(travelers, services)

print("Utility Matrix Shape:", utilities.shape)
print("Utilities (Rows: Travelers, Cols: TNC, MT, MaaS):\n", utilities)

# Check gradients analytically vs autograd
gradients_match = check_gradients(travelers, services, utilities)
print(f"\nDo analytical gradients match Autograd computations? : {gradients_match}")